# PIGNet (Physics-Informed GNN) 虚拟筛选教程

## 论文引用

Moon, S., Zhung, W., Yang, S., Lim, J., & Kim, W. Y. (2022). **PIGNet: A physics-informed deep learning model toward generalized drug-target interaction predictions.** *Chemical Science*, 13, 3661-3673.

## 核心创新

PIGNet 的核心创新是将**物理先验知识**融入深度学习框架，通过**物理信息能量分解 (Physics-Informed Energy Decomposition)** 将蛋白质-配体的结合自由能分解为四种具有明确物理意义的相互作用项：

| 相互作用项 | 物理描述 | 函数形式 |
|-----------|---------|----------|
| **范德华力 (vdW)** | 原子间非键相互作用 | Lennard-Jones 势函数 |
| **氢键 (H-bond)** | 供体-受体间方向性静电作用 | 距离依赖的阶梯函数 |
| **金属配位 (Metal)** | 金属离子与配位原子的相互作用 | 同氢键函数形式 |
| **疏水作用 (Hydrophobic)** | 非极性基团间溶剂排斥效应 | 距离依赖函数 |

每种相互作用使用**可学习的物理函数形式**（而非通用 MLP），兼顾了物理可解释性和数据驱动的灵活性。

## 学习目标

1. 理解 PIGNet 如何将物理先验约束嵌入神经网络架构
2. 掌握蛋白质-配体相互作用的四种基本物理力
3. 学会使用可学习物理参数替代硬编码常数
4. 理解构象熵罚分在结合自由能中的作用
5. 对比物理信息模型与纯黑盒模型的优劣

In [ ]:
# ============================================================
# 项目路径设置与依赖导入
# ============================================================

def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    from pathlib import Path
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'数据目录:   {DATA_DIR}')

import os
import warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem, RDLogger
from rdkit.Chem.rdMolDescriptors import CalcNumRotatableBonds
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

# 关闭 RDKit 警告信息，避免大量输出干扰
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')

# 版本信息
version_info = pd.DataFrame({
    '库': ['PyTorch', 'RDKit', 'NumPy', 'scikit-learn', 'pandas'],
    '版本': [
        torch.__version__,
        Chem.rdBase.rdkitVersion,
        np.__version__,
        __import__('sklearn').__version__,
        pd.__version__,
    ]
})
display(version_info)

## 1. 超参数设置

PIGNet 的超参数分为三类：

1. **物理常数**：Lennard-Jones 势的指数 `VDW_N`、vdW 相互作用强度上下界、半径偏差范围
2. **化学知识**：元素列表、金属离子、疏水原子定义、vdW 半径表、SMARTS 模式
3. **训练参数**：学习率、epoch 数、隐藏层维度

In [ ]:
# ============================================================
# 物理常数与模型超参数
# ============================================================

# --- Lennard-Jones 势参数 ---
# VDW_N: LJ 势中的指数，标准 12-6 势用 N=6
VDW_N = 6
# 范德华相互作用强度的上下界 (kcal/mol)
MAX_VDW_INTERACTION = 0.0356
MIN_VDW_INTERACTION = 0.0178
# 范德华半径偏差的最大允许值 (Angstrom)
DEV_VDW_RADIUS = 0.2

# --- 原子特征与网络维度 ---
# 简化的原子特征: 9维元素 one-hot + 1维芳香性 = 10维
ATOM_FEAT_DIM = 10
HIDDEN_DIM = 128

# --- 训练超参数 ---
N_EPOCHS = 200
LR = 3e-4
BATCH_SIZE = 1      # PIGNet 使用 batch_size=1，因为每个复合物大小不同
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 化学常量定义 ---
# 元素符号列表，"X" 表示其他未列出的元素
SYMBOLS = ["C", "N", "O", "S", "F", "P", "Cl", "Br", "X"]
# 常见金属离子列表（蛋白质中可能出现的配位金属）
METALS = ["Zn", "Mn", "Co", "Mg", "Ni", "Fe", "Ca", "Cu"]
# 疏水性卤素原子
HYDROPHOBICS = ["F", "CL", "BR", "I"]

# 各元素的范德华半径 (Angstrom)，键值为原子序数
VDWRADII = {
    6: 1.90,   # C (碳)
    7: 1.8,    # N (氮)
    8: 1.7,    # O (氧)
    16: 2.0,   # S (硫)
    15: 2.1,   # P (磷)
    9: 1.5,    # F (氟)
    17: 1.8,   # Cl (氯)
    35: 2.0,   # Br (溴)
    53: 2.2,   # I (碘)
    30: 1.2,   # Zn (锌)
    25: 1.2,   # Mn (锰)
    26: 1.2,   # Fe (铁)
    27: 1.2,   # Co (钴)
    12: 1.2,   # Mg (镁)
    28: 1.2,   # Ni (镍)
    20: 1.2,   # Ca (钙)
    29: 1.2,   # Cu (铜)
}

# SMARTS 模式用于识别氢键供体和受体
# 氢键供体: 非碳原子且连有氢原子 (如 -OH, -NH)
HBOND_DONOR_INDICES = ["[!#6;!H0]"]
# 氢键受体: 带孤对电子的非碳原子，排除卤素和特殊情况
HBOND_ACCEPPTOR_SMARTS = [
    "[$([!#6;+0]);!$([F,Cl,Br,I]);!$([o,s,nX3]);!$([Nv5,Pv5,Sv4,Sv6])]"
]

# 打印超参数汇总
params_df = pd.DataFrame({
    '参数': ['VDW_N', 'ATOM_FEAT_DIM', 'HIDDEN_DIM', 'N_EPOCHS', 'LR', 'BATCH_SIZE', 'DEVICE'],
    '值': [VDW_N, ATOM_FEAT_DIM, HIDDEN_DIM, N_EPOCHS, LR, BATCH_SIZE, str(DEVICE)],
    '说明': [
        'Lennard-Jones 势指数 (12-6 势)',
        '原子特征维度 (9 元素 + 1 芳香性)',
        '隐藏层维度',
        '训练轮数',
        '学习率',
        '批大小 (每个复合物大小不同)',
        '计算设备'
    ]
})
display(params_df)

## 2. 数据加载与特征提取

PIGNet 的输入特征分为两大类：

### 原子级特征
- **原子特征** (10维): 9维元素 one-hot 编码 + 1维芳香性标志
- **3D 坐标**: 配体和蛋白质口袋的原子空间位置
- **范德华半径**: 每个原子的 vdW 半径，决定原子间平衡距离

### 相互作用矩阵
- **氢键矩阵** (`N_lig x N_prot`): 基于 SMARTS 模式识别供体-受体对
- **金属配位矩阵**: 金属离子与配位原子的配对
- **疏水矩阵**: 非极性原子对的外积
- **非金属掩码**: vdW 力只适用于非金属原子对

In [ ]:
# ============================================================
# 特征提取函数
# ============================================================

def parse_coreset(path):
    """
    解析 CoreSet.dat 文件，提取 PDB ID 和对应的结合亲和力 (logKa)。

    CoreSet.dat 格式: 第一行以 '#' 开头为表头，之后每行包含:
      列0=pdbid, 列1=分辨率, 列2=年份, 列3=logKa, 列4=Ka值, 列5=靶标簇ID

    返回:
        dict: {pdbid: logKa} 的映射字典
    """
    pdb_to_logka = {}
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 4:
                continue
            pdbid = parts[0]
            try:
                logka = float(parts[3])
            except ValueError:
                continue
            pdb_to_logka[pdbid] = logka
    return pdb_to_logka


def atom_features(atom):
    """
    计算单个原子的特征向量 (10维)。

    特征组成:
      - 9维: 元素类型的 one-hot 编码 (C, N, O, S, F, P, Cl, Br, 其他)
      - 1维: 是否为芳香原子 (布尔值)
    """
    symbol = atom.GetSymbol()
    # 元素 one-hot 编码，未知元素归入 "X" 类
    if symbol not in SYMBOLS[:-1]:
        symbol = "X"
    one_hot = [1.0 if symbol == s else 0.0 for s in SYMBOLS]
    # 芳香性标志
    is_aromatic = [1.0 if atom.GetIsAromatic() else 0.0]
    return np.array(one_hot + is_aromatic, dtype=np.float32)


def get_vdw_radius(atom):
    """
    获取原子的范德华半径。
    优先使用预定义值，不存在则回退到 RDKit 周期表默认值。
    """
    atomic_num = atom.GetAtomicNum()
    if atomic_num in VDWRADII:
        return VDWRADII[atomic_num]
    return Chem.GetPeriodicTable().GetRvdw(atomic_num)


def get_hbond_atom_indices(mol, smarts_list):
    """
    使用 SMARTS 模式匹配找出分子中的氢键供体/受体原子索引。
    """
    indices = []
    for smarts in smarts_list:
        pat = Chem.MolFromSmarts(smarts)
        if pat is None:
            continue
        matches = mol.GetSubstructMatches(pat)
        indices += [match[0] for match in matches]
    return np.array(indices, dtype=np.int64)


def get_hbond_matrix(lig_mol, prot_mol):
    """
    构建氢键相互作用矩阵 (N_lig x N_prot)。

    氢键形成条件: 配体的受体原子 <-> 蛋白质的供体原子
                   或 配体的供体原子 <-> 蛋白质的受体原子
    """
    lig_acc = get_hbond_atom_indices(lig_mol, HBOND_ACCEPPTOR_SMARTS)
    prot_acc = get_hbond_atom_indices(prot_mol, HBOND_ACCEPPTOR_SMARTS)
    lig_don = get_hbond_atom_indices(lig_mol, HBOND_DONOR_INDICES)
    prot_don = get_hbond_atom_indices(prot_mol, HBOND_DONOR_INDICES)

    mat = np.zeros((lig_mol.GetNumAtoms(), prot_mol.GetNumAtoms()), dtype=np.float32)
    for i in lig_acc:
        for j in prot_don:
            mat[i, j] = 1.0
    for i in lig_don:
        for j in prot_acc:
            mat[i, j] = 1.0
    return mat


def get_metal_matrix(lig_mol, prot_mol):
    """
    构建金属配位相互作用矩阵 (N_lig x N_prot)。

    金属配位: 金属离子与含孤对电子的原子(氢键受体)之间的配位键。

    注意: 原始 PIGNet 代码中此函数有一个 bug —— 使用了变量 `i` 而非 `idx` 来索引原子。
    本实现已修复此 bug，正确使用 `idx` 进行索引。
    """
    lig_acc = get_hbond_atom_indices(lig_mol, HBOND_ACCEPPTOR_SMARTS)
    prot_acc = get_hbond_atom_indices(prot_mol, HBOND_ACCEPPTOR_SMARTS)

    # 修复了原始代码的 bug: GetAtomWithIdx(i) -> GetAtomWithIdx(idx)
    lig_metal = np.array([
        idx for idx in range(lig_mol.GetNumAtoms())
        if lig_mol.GetAtomWithIdx(idx).GetSymbol() in METALS
    ], dtype=np.int64)
    prot_metal = np.array([
        idx for idx in range(prot_mol.GetNumAtoms())
        if prot_mol.GetAtomWithIdx(idx).GetSymbol() in METALS
    ], dtype=np.int64)

    mat = np.zeros((lig_mol.GetNumAtoms(), prot_mol.GetNumAtoms()), dtype=np.float32)
    # 配体受体 <-> 蛋白质金属
    for i in lig_acc:
        for j in prot_metal:
            mat[i, j] = 1.0
    # 配体金属 <-> 蛋白质受体
    for i in lig_metal:
        for j in prot_acc:
            mat[i, j] = 1.0
    return mat


def get_hydrophobic_atom(mol):
    """
    识别分子中的疏水性原子。

    疏水性原子包括:
      1. 卤素原子 (F, Cl, Br, I)
      2. 仅与碳相连的碳原子 (纯碳骨架区域)
    """
    natoms = mol.GetNumAtoms()
    hydrophobic = np.zeros((natoms,), dtype=np.float32)
    for i in range(natoms):
        atom = mol.GetAtomWithIdx(i)
        sym = atom.GetSymbol()
        if sym.upper() in HYDROPHOBICS:
            hydrophobic[i] = 1.0
        elif sym.upper() == "C":
            neighbors = [x.GetSymbol() for x in atom.GetNeighbors()]
            # 只有当所有邻居都是碳时，才认为是疏水碳
            if len(neighbors) > 0 and all(n == "C" for n in neighbors):
                hydrophobic[i] = 1.0
    return hydrophobic


def get_hydrophobic_matrix(lig_mol, prot_mol):
    """
    构建疏水相互作用矩阵 (N_lig x N_prot)。
    矩阵是两个疏水原子向量的外积: mat[i,j] = lig_hydro[i] * prot_hydro[j]
    """
    lig_hydro = get_hydrophobic_atom(lig_mol)
    prot_hydro = get_hydrophobic_atom(prot_mol)
    return np.outer(lig_hydro, prot_hydro).astype(np.float32)


def get_non_metal_mask(mol):
    """
    获取非金属原子的掩码向量。
    vdW 力只适用于非金属原子对，金属原子使用配位相互作用项描述。
    """
    mask = np.array([
        1.0 if atom.GetSymbol() not in METALS else 0.0
        for atom in mol.GetAtoms()
    ], dtype=np.float32)
    return mask


def load_complex_pignet(pdbid):
    """
    加载单个蛋白质-配体复合物的所有特征。

    加载过程:
      1. 读取口袋 PDB 文件和配体 MOL2/SDF 文件
      2. 移除氢原子 (简化计算)
      3. 提取原子特征、3D 坐标、范德华半径
      4. 计算氢键/金属配位/疏水相互作用矩阵
      5. 统计可旋转键数目 (用于熵罚分)

    返回:
        dict: 包含所有特征的字典，失败时返回 None
    """
    complex_dir = COMPLEX_DIR / pdbid
    pocket_file = complex_dir / f"{pdbid}_pocket.pdb"
    ligand_mol2 = complex_dir / f"{pdbid}_ligand.mol2"
    ligand_sdf = complex_dir / f"{pdbid}_ligand.sdf"

    # 加载蛋白质口袋
    # sanitize=False: 跳过化学合理性检查，避免 PDB 文件中常见的化学价问题
    prot_mol = Chem.MolFromPDBFile(str(pocket_file), sanitize=False, removeHs=True)
    if prot_mol is None:
        return None
    try:
        prot_mol = Chem.RemoveHs(prot_mol)
    except Exception:
        pass

    # 加载配体分子，优先使用 MOL2 格式，回退到 SDF
    lig_mol = None
    if ligand_mol2.exists():
        lig_mol = Chem.MolFromMol2File(str(ligand_mol2), sanitize=False)
    if lig_mol is None and ligand_sdf.exists():
        supplier = Chem.SDMolSupplier(str(ligand_sdf), sanitize=False)
        for mol in supplier:
            if mol is not None:
                lig_mol = mol
                break
    if lig_mol is None:
        return None
    try:
        lig_mol = Chem.RemoveHs(lig_mol)
    except Exception:
        pass

    # 检查 3D 坐标是否存在
    if lig_mol.GetNumConformers() == 0 or prot_mol.GetNumConformers() == 0:
        return None

    n_lig = lig_mol.GetNumAtoms()
    n_prot = prot_mol.GetNumAtoms()
    if n_lig == 0 or n_prot == 0:
        return None

    # 提取原子特征 (10维)
    lig_h = np.array([atom_features(lig_mol.GetAtomWithIdx(i))
                      for i in range(n_lig)], dtype=np.float32)
    prot_h = np.array([atom_features(prot_mol.GetAtomWithIdx(i))
                       for i in range(n_prot)], dtype=np.float32)

    # 提取 3D 坐标
    lig_pos = np.array(lig_mol.GetConformers()[0].GetPositions(), dtype=np.float32)
    prot_pos = np.array(prot_mol.GetConformers()[0].GetPositions(), dtype=np.float32)

    # 范德华半径
    lig_vdw = np.array([get_vdw_radius(lig_mol.GetAtomWithIdx(i))
                        for i in range(n_lig)], dtype=np.float32)
    prot_vdw = np.array([get_vdw_radius(prot_mol.GetAtomWithIdx(i))
                         for i in range(n_prot)], dtype=np.float32)

    # 相互作用矩阵
    hbond_mat = get_hbond_matrix(lig_mol, prot_mol)
    metal_mat = get_metal_matrix(lig_mol, prot_mol)
    hydrophobic_mat = get_hydrophobic_matrix(lig_mol, prot_mol)

    # 非金属原子掩码 (vdW 力只作用于非金属原子对)
    lig_non_metal = get_non_metal_mask(lig_mol)
    prot_non_metal = get_non_metal_mask(prot_mol)

    # 可旋转键数目: 用于计算构象熵罚分
    try:
        rotor = CalcNumRotatableBonds(lig_mol)
    except Exception:
        rotor = 0

    sample = {
        'lig_h': lig_h,                       # (N_l, 10) 配体原子特征
        'prot_h': prot_h,                      # (N_p, 10) 蛋白质原子特征
        'lig_pos': lig_pos,                    # (N_l, 3)  配体原子坐标
        'prot_pos': prot_pos,                  # (N_p, 3)  蛋白质原子坐标
        'lig_vdw': lig_vdw,                    # (N_l,)    配体 vdW 半径
        'prot_vdw': prot_vdw,                  # (N_p,)    蛋白质 vdW 半径
        'hbond_mat': hbond_mat,                # (N_l, N_p) 氢键矩阵
        'metal_mat': metal_mat,                # (N_l, N_p) 金属配位矩阵
        'hydrophobic_mat': hydrophobic_mat,    # (N_l, N_p) 疏水矩阵
        'lig_non_metal': lig_non_metal,        # (N_l,)    配体非金属掩码
        'prot_non_metal': prot_non_metal,      # (N_p,)    蛋白质非金属掩码
        'rotor': float(rotor),                 # 标量: 可旋转键数
    }
    return sample

In [ ]:
# ============================================================
# 加载数据并展示一个样本
# ============================================================

# 设置随机种子
torch.manual_seed(SEED)
np.random.seed(SEED)

# 加载结合亲和力标签
print("[1/4] 正在加载结合亲和力标签...")
pdb_to_logka = parse_coreset(CORESET_FILE)
print(f"  读取到 {len(pdb_to_logka)} 个 PDB 条目")

# 特征提取
print("[2/4] 正在提取蛋白质-配体复合物特征...")
all_data = []
failed = 0
for i, (pdbid, logka) in enumerate(pdb_to_logka.items()):
    sample = load_complex_pignet(pdbid)
    if sample is not None:
        sample['label'] = logka
        all_data.append(sample)
    else:
        failed += 1
    if (i + 1) % 50 == 0:
        print(f"  已处理 {i + 1}/{len(pdb_to_logka)} ... (成功: {len(all_data)}, 失败: {failed})")

print(f"  特征提取完成: {len(all_data)} 个成功, {failed} 个跳过")

if len(all_data) < 10:
    raise RuntimeError("有效复合物数量不足，请检查数据路径。")

# 展示一个样本的特征形状
sample_0 = all_data[0]
sample_info = pd.DataFrame({
    '特征': list(sample_0.keys()),
    '形状/值': [
        str(v.shape) if isinstance(v, np.ndarray) else str(v)
        for v in sample_0.values()
    ],
    '说明': [
        '配体原子特征 (元素 one-hot + 芳香性)',
        '蛋白质原子特征',
        '配体原子 3D 坐标',
        '蛋白质原子 3D 坐标',
        '配体原子 vdW 半径',
        '蛋白质原子 vdW 半径',
        '氢键相互作用矩阵',
        '金属配位相互作用矩阵',
        '疏水相互作用矩阵',
        '配体非金属原子掩码',
        '蛋白质非金属原子掩码',
        '可旋转键数目',
        '结合亲和力 logKa',
    ]
})
print("\n第一个样本的特征概览:")
display(sample_info)

## 3. 数据集与数据加载器

由于每个蛋白质-配体复合物的原子数不同（配体通常 10-80 个原子，口袋 100-500 个原子），张量形状各不相同，无法在 batch 维度上直接合并。因此 PIGNet 使用 `batch_size=1`，每次只处理一个复合物。

数据集按 80/20 比例划分为训练集和验证集。

In [ ]:
# ============================================================
# 数据集类与 DataLoader
# ============================================================

class PIGNetDataset(Dataset):
    """
    PIGNet 数据集类。

    每个样本是一个蛋白质-配体复合物，包含:
    - 原子特征、坐标、范德华半径
    - 氢键/金属/疏水相互作用矩阵
    - 非金属掩码、可旋转键数
    - 结合亲和力标签 (logKa)
    """
    def __init__(self, data_list):
        self.data_list = data_list

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        d = self.data_list[idx]
        sample = {
            'lig_h': torch.tensor(d['lig_h'], dtype=torch.float32),
            'prot_h': torch.tensor(d['prot_h'], dtype=torch.float32),
            'lig_pos': torch.tensor(d['lig_pos'], dtype=torch.float32),
            'prot_pos': torch.tensor(d['prot_pos'], dtype=torch.float32),
            'lig_vdw': torch.tensor(d['lig_vdw'], dtype=torch.float32),
            'prot_vdw': torch.tensor(d['prot_vdw'], dtype=torch.float32),
            'hbond_mat': torch.tensor(d['hbond_mat'], dtype=torch.float32),
            'metal_mat': torch.tensor(d['metal_mat'], dtype=torch.float32),
            'hydrophobic_mat': torch.tensor(d['hydrophobic_mat'], dtype=torch.float32),
            'lig_non_metal': torch.tensor(d['lig_non_metal'], dtype=torch.float32),
            'prot_non_metal': torch.tensor(d['prot_non_metal'], dtype=torch.float32),
            'rotor': torch.tensor(d['rotor'], dtype=torch.float32),
            'label': torch.tensor(d['label'], dtype=torch.float32),
        }
        return sample


# 划分数据集
print("[3/4] 划分训练/验证集 (80%/20%)...")
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_data = [all_data[i] for i in indices[:split]]
val_data = [all_data[i] for i in indices[split:]]

train_dataset = PIGNetDataset(train_data)
val_dataset = PIGNetDataset(val_data)

# batch_size=1: 每个复合物原子数不同，无法在 batch 维度上合并
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 展示数据集统计
split_df = pd.DataFrame({
    '数据集': ['训练集', '验证集', '总计'],
    '样本数': [len(train_data), len(val_data), len(all_data)],
    '占比': [f'{len(train_data)/len(all_data)*100:.0f}%',
            f'{len(val_data)/len(all_data)*100:.0f}%',
            '100%']
})
display(split_df)

# 展示一个 batch 的张量形状
sample_batch = next(iter(train_loader))
print("\n单个 batch 的张量形状:")
shapes_df = pd.DataFrame({
    '键': list(sample_batch.keys()),
    '形状': [str(tuple(v.shape)) for v in sample_batch.values()]
})
display(shapes_df)

## 4. 模型架构

PIGNet 的核心设计思想是**用物理函数形式替代通用 MLP**，实现物理可解释的能量预测。

### 物理信息能量分解

总结合能由四项组成，每项使用特定的物理函数形式：

#### 1. 范德华力 (Lennard-Jones 势)
$$E_{\text{vdW}} = A \cdot \left[ \left(\frac{d_{m_0}}{d_m}\right)^{2N} - 2\left(\frac{d_{m_0}}{d_m}\right)^N \right]$$

其中 $A$ 为可学习的相互作用强度，$d_{m_0} = r_{\text{vdW,lig}} + r_{\text{vdW,prot}} + B$ 为可学习的平衡距离。

#### 2. 氢键与金属配位
基于距离的线性衰减函数，通过 SMARTS 模式矩阵控制哪些原子对可以形成氢键/配位键。

#### 3. 疏水作用
非极性基团间的距离依赖函数，有效范围比 vdW 平衡距离更大 (+1.5 A)。

### 构象熵罚分
$$\Delta G_{\text{bind}} = \frac{E_{\text{total}}}{1 + \alpha \cdot n_{\text{rotor}}}$$

可旋转键越多的配体，结合时构象自由度损失越大。

In [ ]:
# ============================================================
# 模型架构: PIGNetToyModel
# ============================================================

class PIGNetToyModel(nn.Module):
    """
    PIGNet 简化版模型: 基于物理信息的蛋白质-配体结合亲和力预测。

    核心设计思想:
    ----------------
    1. 物理先验约束: 每种相互作用使用特定的物理函数形式
    2. 可学习物理参数: 势函数参数由神经网络预测
    3. 能量分解: 总结合能 = vdW + 氢键 + 金属配位 + 疏水
    4. 构象熵罚分: 通过可旋转键数目施加熵惩罚
    """
    def __init__(self, atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        # 原子特征嵌入层: 将 10 维化学特征映射到高维隐空间
        self.node_embed = nn.Linear(atom_dim, hidden_dim)

        # ---- 范德华参数预测网络 ----
        # cal_vdw_A: 预测每对原子的 vdW 相互作用强度 A
        # Sigmoid 确保 A in [0, 1]，后续缩放到 [MIN_VDW, MAX_VDW]
        self.cal_vdw_A = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Sigmoid()
        )
        # cal_vdw_B: 预测范德华半径的偏差 B
        # Tanh 确保 B in [-1, 1]，后续缩放到 [-DEV_VDW_RADIUS, DEV_VDW_RADIUS]
        self.cal_vdw_B = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Tanh()
        )

        # ---- 可学习物理系数 ----
        # 氢键强度系数 (使用平方确保为正)
        self.hbond_coeff = nn.Parameter(torch.tensor([1.0]))
        # 疏水相互作用强度系数
        self.hydrophobic_coeff = nn.Parameter(torch.tensor([0.5]))
        # 可旋转键熵罚分系数
        self.rotor_coeff = nn.Parameter(torch.tensor([0.5]))

        # 输出缩放层: 将物理能量映射到 logKa 尺度
        self.output_scale = nn.Parameter(torch.tensor([-1.0]))
        self.output_bias = nn.Parameter(torch.tensor([6.0]))

    def forward(self, sample):
        """
        前向传播: 计算蛋白质-配体结合能量。

        物理意义:
          结合自由能 dG = (E_vdw + E_hbond + E_metal + E_hydro) / (1 + alpha * n_rotor)

        Returns:
            total: 预测的总结合能
            vdw_energy: 范德华能量贡献
            hbond_energy: 氢键能量贡献
            metal_energy: 金属配位能量贡献
            hydro_energy: 疏水能量贡献
        """
        # 原子特征嵌入
        lig_h = self.node_embed(sample['lig_h'])      # (N_l, hidden)
        prot_h = self.node_embed(sample['prot_h'])     # (N_p, hidden)

        # 计算距离矩阵 dm[i,j] = ||lig_pos[i] - prot_pos[j]||
        dm = torch.cdist(
            sample['lig_pos'].unsqueeze(0),
            sample['prot_pos'].unsqueeze(0)
        ).squeeze(0)
        dm = dm.clamp(min=0.5)  # 避免除零

        # 距离截断掩码: 只考虑 10A 以内的原子对
        distance_mask = (dm < 10.0).float()

        # 构建成对原子特征
        N_l, N_p = lig_h.size(0), prot_h.size(0)
        lig_expand = lig_h.unsqueeze(1).expand(-1, N_p, -1)    # (N_l, N_p, hidden)
        prot_expand = prot_h.unsqueeze(0).expand(N_l, -1, -1)  # (N_l, N_p, hidden)
        h_cat = torch.cat([lig_expand, prot_expand], dim=-1)    # (N_l, N_p, hidden*2)

        # 范德华半径: 平衡距离 dm_0 = r_vdw_lig + r_vdw_prot + B
        lig_vdw = sample['lig_vdw'].unsqueeze(1).expand(-1, N_p)
        prot_vdw = sample['prot_vdw'].unsqueeze(0).expand(N_l, -1)

        B = self.cal_vdw_B(h_cat).squeeze(-1) * DEV_VDW_RADIUS
        dm_0 = lig_vdw + prot_vdw + B
        dm_0 = dm_0.clamp(min=0.0001)

        # ================================================================
        # 1. 范德华能量 (van der Waals Energy)
        # ================================================================
        # Lennard-Jones 势: E_vdw = A * [(dm0/dm)^{2N} - 2*(dm0/dm)^N]
        A = self.cal_vdw_A(h_cat).squeeze(-1)
        A = A * (MAX_VDW_INTERACTION - MIN_VDW_INTERACTION) + MIN_VDW_INTERACTION
        ratio = dm_0 / dm
        vdw_energy = A * (ratio.pow(2 * VDW_N) - 2 * ratio.pow(VDW_N))
        vdw_energy = vdw_energy.clamp(max=100)  # 防止近距离排斥项爆炸

        # 非金属掩码: vdW 力只适用于非金属原子对
        non_metal_mask = (sample['lig_non_metal'].unsqueeze(1)
                          * sample['prot_non_metal'].unsqueeze(0))
        vdw_energy = (vdw_energy * non_metal_mask * distance_mask).sum()

        # ================================================================
        # 2. 氢键能量 (Hydrogen Bond Energy)
        # ================================================================
        # 基于距离的线性衰减函数
        # -0.7 Angstrom 是氢键的特征距离尺度
        dm_shift = dm - dm_0
        hbond_term = (dm_shift / -0.7).clamp(0, 1) * sample['hbond_mat'] * distance_mask
        hbond_energy = (hbond_term * -(self.hbond_coeff ** 2)).sum()

        # ================================================================
        # 3. 金属配位能量 (Metal Coordination Energy)
        # ================================================================
        # 使用与氢键相同的函数形式，作用于金属-配位原子对
        metal_term = (dm_shift / -0.7).clamp(0, 1) * sample['metal_mat'] * distance_mask
        metal_energy = (metal_term * -(self.hbond_coeff ** 2)).sum()

        # ================================================================
        # 4. 疏水能量 (Hydrophobic Energy)
        # ================================================================
        # +1.5 Angstrom: 疏水作用的有效范围比 vdW 平衡距离更大
        hydro_term = (-dm + dm_0 + 1.5).clamp(0, 1) * sample['hydrophobic_mat'] * distance_mask
        hydro_energy = (hydro_term * -(self.hydrophobic_coeff ** 2)).sum()

        # ================================================================
        # 总能量与构象熵罚分
        # ================================================================
        total = vdw_energy + hbond_energy + metal_energy + hydro_energy
        # 构象熵罚分: 可旋转键越多，结合时熵损失越大
        total = total / (1 + self.rotor_coeff ** 2 * sample['rotor'])

        # 将物理能量映射到 logKa 尺度
        total = self.output_scale * total + self.output_bias

        return (total,
                vdw_energy.detach(),
                hbond_energy.detach(),
                metal_energy.detach(),
                hydro_energy.detach())

In [ ]:
# ============================================================
# 模型实例化与参数统计
# ============================================================

model = PIGNetToyModel(atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型总参数量: {n_params:,}")
print(f"\n模型结构:\n{model}")

# 各层参数量统计
layer_params = []
for name, param in model.named_parameters():
    layer_params.append({
        '层名': name,
        '形状': str(tuple(param.shape)),
        '参数量': param.numel(),
        '可训练': param.requires_grad
    })
params_detail_df = pd.DataFrame(layer_params)
display(params_detail_df)

## 5. 训练

训练过程使用 MSE 损失函数和 Adam 优化器。每 20 个 epoch 打印：
- 训练/验证损失
- 各能量分量的平均贡献（监控模型是否学到合理的物理分解）

梯度裁剪 (`max_norm=5.0`) 防止物理能量计算中的梯度爆炸。

In [ ]:
# ============================================================
# 训练循环
# ============================================================

def train_model(model, train_loader, val_loader, n_epochs=N_EPOCHS, lr=LR):
    """
    训练 PIGNet 模型。

    Returns:
        train_losses: 各 epoch 的训练损失
        val_losses: 各 epoch 的验证损失
        energy_records: 各能量分量的记录
    """
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.to(DEVICE)

    train_losses = []
    val_losses = []
    energy_records = {'vdw': [], 'hbond': [], 'metal': [], 'hydrophobic': []}

    print("=" * 80)
    print("PIGNet Toy Model -- Training")
    print(f"Device: {DEVICE} | Train: {len(train_loader)} | Val: {len(val_loader)}")
    print(f"Hidden dim: {HIDDEN_DIM} | LR: {lr} | Epochs: {n_epochs}")
    print("=" * 80)

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_vdw, epoch_hbond, epoch_metal, epoch_hydro = 0.0, 0.0, 0.0, 0.0
        n_train = 0

        for batch in train_loader:
            # 将数据移到设备上
            sample = {k: v.squeeze(0).to(DEVICE) if isinstance(v, torch.Tensor) else v
                      for k, v in batch.items()}
            label = sample.pop('label').to(DEVICE)

            optimizer.zero_grad()
            pred, vdw_e, hbond_e, metal_e, hydro_e = model(sample)
            loss = criterion(pred, label)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            epoch_loss += loss.item()
            epoch_vdw += vdw_e.item()
            epoch_hbond += hbond_e.item()
            epoch_metal += metal_e.item()
            epoch_hydro += hydro_e.item()
            n_train += 1

        avg_train_loss = epoch_loss / max(n_train, 1)
        train_losses.append(avg_train_loss)
        energy_records['vdw'].append(epoch_vdw / max(n_train, 1))
        energy_records['hbond'].append(epoch_hbond / max(n_train, 1))
        energy_records['metal'].append(epoch_metal / max(n_train, 1))
        energy_records['hydrophobic'].append(epoch_hydro / max(n_train, 1))

        # 验证
        model.train(False)
        val_loss = 0.0
        n_val = 0
        with torch.no_grad():
            for batch in val_loader:
                sample = {k: v.squeeze(0).to(DEVICE)
                          if isinstance(v, torch.Tensor) else v
                          for k, v in batch.items()}
                label = sample.pop('label').to(DEVICE)
                pred, _, _, _, _ = model(sample)
                loss = criterion(pred, label)
                val_loss += loss.item()
                n_val += 1

        avg_val_loss = val_loss / max(n_val, 1)
        val_losses.append(avg_val_loss)

        # 每 20 个 epoch 输出详细信息
        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1:>4d}/{n_epochs} | "
                  f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
                  f"vdW: {energy_records['vdw'][-1]:.4f} | HBond: {energy_records['hbond'][-1]:.4f} | "
                  f"Metal: {energy_records['metal'][-1]:.4f} | Hydro: {energy_records['hydrophobic'][-1]:.4f}")

    print("=" * 80)
    print("Training complete!")
    print("=" * 80)
    return train_losses, val_losses, energy_records


print("[4/4] 开始训练模型...")
train_losses, val_losses, energy_records = train_model(
    model, train_loader, val_loader, n_epochs=N_EPOCHS, lr=LR
)

## 6. 评估与可视化

评估指标：
- **Pearson R**: 线性相关系数，衡量预测值与真实值的线性关系
- **RMSE**: 均方根误差，衡量预测的绝对精度

可视化包含：
1. **散点图**: 预测 logKa vs 真实 logKa，附拟合线和对角线
2. **能量分解图**: 箱线图展示各能量分量在测试集上的分布 + 折线图展示训练过程中各能量分量的变化

In [ ]:
# ============================================================
# 测试集评估
# ============================================================

model.train(False)
model.to(DEVICE)

true_values = []
pred_values = []
all_vdw, all_hbond, all_metal, all_hydro = [], [], [], []

with torch.no_grad():
    for batch in val_loader:
        sample = {k: v.squeeze(0).to(DEVICE)
                  if isinstance(v, torch.Tensor) else v
                  for k, v in batch.items()}
        label = sample.pop('label').to(DEVICE)
        pred, vdw_e, hbond_e, metal_e, hydro_e = model(sample)

        true_values.append(label.item())
        pred_values.append(pred.item())
        all_vdw.append(vdw_e.item())
        all_hbond.append(hbond_e.item())
        all_metal.append(metal_e.item())
        all_hydro.append(hydro_e.item())

true_arr = np.array(true_values)
pred_arr = np.array(pred_values)

# 计算评估指标
r, p_value = pearsonr(true_arr, pred_arr)
rmse = np.sqrt(mean_squared_error(true_arr, pred_arr))

# 评估结果表格
eval_df = pd.DataFrame({
    '指标': ['测试样本数', 'Pearson R', 'p-value', 'RMSE'],
    '值': [len(true_arr), f'{r:.4f}', f'{p_value:.2e}', f'{rmse:.4f}']
})
print("测试集评估结果:")
display(eval_df)

In [ ]:
# ============================================================
# 可视化
# ============================================================

# ---- 图1: 预测值 vs 真实值散点图 ----
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
ax.scatter(true_arr, pred_arr, alpha=0.6, edgecolors='k',
           linewidths=0.5, s=40, c='steelblue')

# 拟合线
if len(true_arr) > 1:
    z = np.polyfit(true_arr, pred_arr, 1)
    p = np.poly1d(z)
    x_line = np.linspace(true_arr.min(), true_arr.max(), 100)
    ax.plot(x_line, p(x_line), 'r--', linewidth=1.5,
            label=f'Fit: y={z[0]:.2f}x+{z[1]:.2f}')

# 对角线 (理想预测)
lims = [min(true_arr.min(), pred_arr.min()) - 0.5,
        max(true_arr.max(), pred_arr.max()) + 0.5]
ax.plot(lims, lims, 'k:', alpha=0.3, linewidth=1)
ax.set_xlim(lims[0], lims[1])
ax.set_ylim(lims[0], lims[1])

ax.set_xlabel('True logKa', fontsize=13)
ax.set_ylabel('Predicted logKa', fontsize=13)
ax.set_title(f'PIGNet Toy Model\nPearson R = {r:.3f}, RMSE = {rmse:.3f}', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# ---- 图2: 能量分解可视化 ----
fig2, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 箱线图 -- 各能量分量在测试集上的分布
energy_data = [all_vdw, all_hbond, all_metal, all_hydro]
labels = ['vdW', 'H-Bond', 'Metal', 'Hydrophobic']
colors = ['#4ECDC4', '#FF6B6B', '#C9B1FF', '#95E1D3']

bp = axes[0].boxplot(energy_data, labels=labels, patch_artist=True,
                     showmeans=True, meanline=True,
                     medianprops=dict(color='black', linewidth=1.5),
                     meanprops=dict(color='red', linewidth=1.5,
                                    linestyle='--'))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_ylabel('Energy (arbitrary units)', fontsize=12)
axes[0].set_title('Energy Decomposition (Test Set)', fontsize=13)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].axhline(y=0, color='gray', linestyle='-', alpha=0.5)

# 右图: 训练过程中各能量分量的变化趋势
epochs_range = range(1, len(energy_records['vdw']) + 1)
axes[1].plot(epochs_range, energy_records['vdw'],
             label='vdW', color='#4ECDC4', linewidth=1.5)
axes[1].plot(epochs_range, energy_records['hbond'],
             label='H-Bond', color='#FF6B6B', linewidth=1.5)
axes[1].plot(epochs_range, energy_records['metal'],
             label='Metal', color='#C9B1FF', linewidth=1.5)
axes[1].plot(epochs_range, energy_records['hydrophobic'],
             label='Hydrophobic', color='#95E1D3', linewidth=1.5)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Average Energy', fontsize=12)
axes[1].set_title('Energy Components During Training', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='gray', linestyle='-', alpha=0.5)

fig2.tight_layout()
plt.show()

## 总结

### PIGNet 的核心教学要点

1. **物理先验 vs 黑盒学习**: PIGNet 的核心创新是将物理先验知识融入深度学习框架。每种能量项使用特定的物理函数形式（如 Lennard-Jones 势），而非通用的黑盒 MLP。

2. **可学习物理参数**: 势函数的参数（如相互作用强度 $A$、平衡距离偏差 $B$）由神经网络预测，使模型既保持物理合理性，又能适应不同化学环境。

3. **能量分解的优势**:
   - **物理可解释性** -- 可以分析每种相互作用（vdW、氢键、金属配位、疏水）对总结合能的贡献
   - **泛化能力** -- 物理约束减少了过拟合风险，提高了对新分子体系的预测能力

4. **构象熵罚分**: 通过可旋转键数目 $n_{\text{rotor}}$ 对总能量施加熵惩罚，反映了配体结合时构象自由度的损失。

5. **本教程简化说明**: 本 toy 模型省略了原始 PIGNet 中的图神经网络消息传递模块，仅保留了物理能量计算的核心部分。完整的 PIGNet 还包括多层 GNN 来更新原子表示，以捕获更丰富的化学环境信息。

### 参考文献

- Moon, S., Zhung, W., Yang, S., Lim, J., & Kim, W. Y. (2022). PIGNet: A physics-informed deep learning model toward generalized drug-target interaction predictions. *Chemical Science*, 13, 3661-3673.